In [ ]:
#@title 🎧 Download Narration Audio & Play Introduction
import os as _os
if not _os.path.exists("/content/narration"):
    !pip install -q gdown
    import gdown
    gdown.download(id="19gb4_BhWWizhtgnXs2oSMzptKIJstJks", output="/content/narration.zip", quiet=False)
    !unzip -q /content/narration.zip -d /content/narration
    !rm /content/narration.zip
    print(f"Loaded {len(_os.listdir('/content/narration'))} narration segments")
else:
    print("Narration audio already loaded.")

from IPython.display import Audio, display
display(Audio("/content/narration/02_00_intro.mp3"))


In [ ]:
#@title 🎧 Code Walkthrough: Setup
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_01_setup.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# 🔧 Setup: Run this cell first!
# Check GPU availability and install dependencies

import torch
import sys

# Check GPU
if torch.cuda.is_available():
    device = torch.device('cuda')
    print(f"✅ GPU available: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    device = torch.device('cpu')
    print("⚠️ No GPU detected. This notebook runs fine on CPU.")
    print("   Go to Runtime → Change runtime type → GPU (optional)")

print(f"\n📦 Python {sys.version.split()[0]}")
print(f"🔥 PyTorch {torch.__version__}")

# Set random seeds for reproducibility
import random
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"🎲 Random seed set to {SEED}")

import matplotlib.pyplot as plt

# Clean plotting style
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': '#fafafa',
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 11,
    'figure.dpi': 100,
})

%matplotlib inline

In [ ]:
#@title 🎧 Listen: Notebook Overview
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_02_notebook_overview.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


# 🧠 DeltaNet: Error-Correcting Memory

*Part 2 of the Vizuara series on Building Qwen3.5 from Scratch*
*Estimated time: ~90 minutes*

In Notebook 1, we saw how linear attention achieves $O(N)$ complexity by rewriting attention as a recurrence with a hidden state $S_t = S_{t-1} + \tilde{k}_t v_t^T$. This is fast, but it has a fatal flaw: the state is purely additive. Old information never gets erased. Over time, the memory fills up with noise, and retrieval degrades.

In this notebook, we will:
1. **See the forgetting problem in action** with a concrete key-value memory example
2. **Implement the delta rule** — a 1960s idea that turns a dumb accumulator into an error-correcting memory
3. **Build a full benchmark** comparing naive linear attention vs DeltaNet on associative retrieval
4. **Visualize why the delta rule works** by examining the state matrix structure
5. **Understand why the delta rule alone is not enough** — motivating the gating mechanism in Notebook 3

In [ ]:
#@title 🎧 Listen: Ai Assistant
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_03_ai_assistant.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


# 🤖 AI Teaching Assistant

Need help with this notebook? Open the **AI Teaching Assistant** — it has already read this entire notebook and can help with concepts, code, and exercises.

**[👉 Open AI Teaching Assistant](https://pods.vizuara.ai/courses/build-llm/practice/7/assistant)**

*Tip: Open it in a separate tab and work through this notebook side-by-side.*

In [ ]:
#@title 🎧 Listen: Forgetting Problem Recap
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_04_forgetting_problem_recap.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 1. 💥 The Forgetting Problem

### Recap: Linear Attention as a Recurrence

In Notebook 1, we derived the linear attention recurrence. At each time step $t$, the hidden state is:

$$S_t = S_{t-1} + \tilde{k}_t v_t^T$$

where $\tilde{k}_t = \phi(k_t)$ is the key after applying a feature map, and $v_t$ is the value vector. The output for a query $\tilde{q}_t$ is simply:

$$\text{output}_t = S_t^T \tilde{q}_t$$

This state $S_t$ is the model's **memory** — a compressed summary of everything it has seen. Each new token writes to this memory by adding its outer product $\tilde{k}_t v_t^T$.

The problem? **Writes are purely additive.** There is no mechanism to erase, correct, or overwrite. Let us see exactly what goes wrong.

In [ ]:
#@title 🎧 Code Walkthrough: Demo Forgetting Orthogonal
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_05_demo_forgetting_orthogonal.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Demonstrate the forgetting problem with a simple key-value memory
print("=" * 60)
print("  THE FORGETTING PROBLEM: Naive Linear Attention Memory")
print("=" * 60)

# Start with a 2D key-value memory
# Keys are unit vectors, values are 2D vectors
d = 2
S = torch.zeros(d, d)  # Empty state matrix

# Step 1: Store "capital of France" -> "Paris"
# We represent this with key k1 and value v_paris
k_france = torch.tensor([1.0, 0.0])  # Key for "France"
v_paris = torch.tensor([0.8, 0.2])   # Value for "Paris"

# Naive additive update
S = S + torch.outer(k_france, v_paris)

print(f"\n--- Step 1: Store France -> Paris ---")
print(f"Key:   {k_france.tolist()}")
print(f"Value: {v_paris.tolist()}")
print(f"State S after write:\n{S}")

# Query: what does the memory say about France?
retrieved = S.T @ k_france
print(f"\nQuery with France key: {retrieved.tolist()}")
print(f"Expected:              {v_paris.tolist()}")
print(f"Match: {'✅ Exact!' if torch.allclose(retrieved, v_paris) else '❌ Mismatch'}")

# Step 2: Also store "capital of Germany" -> "Berlin"
k_germany = torch.tensor([0.0, 1.0])  # Key for "Germany" (orthogonal to France)
v_berlin = torch.tensor([0.3, 0.9])   # Value for "Berlin"

S = S + torch.outer(k_germany, v_berlin)

print(f"\n--- Step 2: Store Germany -> Berlin ---")
print(f"State S after both writes:\n{S}")

# Queries still work with orthogonal keys
retrieved_france = S.T @ k_france
retrieved_germany = S.T @ k_germany
print(f"\nQuery France:  {retrieved_france.tolist()} (expected {v_paris.tolist()})")
print(f"Query Germany: {retrieved_germany.tolist()} (expected {v_berlin.tolist()})")
print(f"Both exact: {'✅' if torch.allclose(retrieved_france, v_paris) and torch.allclose(retrieved_germany, v_berlin) else '❌'}")


In [ ]:
#@title 🎧 Listen: After Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_06_after_code_4.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
print(f"\n✨ With orthogonal keys, additive memory works perfectly.")

In [ ]:
#@title 🎧 Code Walkthrough: Demo Forgetting Update
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_07_demo_forgetting_update.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Now the problem: UPDATE France -> Lyon (same key, new value)
print("=" * 60)
print("  THE PROBLEM: Updating a Value")
print("=" * 60)

v_lyon = torch.tensor([0.6, 0.7])  # New value: "Lyon"

# Naive additive update — just add the new key-value pair
S_naive = S + torch.outer(k_france, v_lyon)

print(f"\n--- Updating France: Paris -> Lyon ---")
print(f"State S after naive update:\n{S_naive}")

# Query France again
retrieved_naive = S_naive.T @ k_france
print(f"\nQuery France:  {retrieved_naive.tolist()}")
print(f"Expected (Lyon): {v_lyon.tolist()}")
print(f"Old (Paris):     {v_paris.tolist()}")

# The retrieved value is Paris + Lyon superposed!
expected_superposition = v_paris + v_lyon
print(f"\nActual retrieval = Paris + Lyon = {expected_superposition.tolist()}")
print(f"Match superposition: {'✅' if torch.allclose(retrieved_naive, expected_superposition) else '❌'}")

print(f"\n❌ PROBLEM: The memory returns Paris + Lyon superposed!")
print(f"   Old value 'Paris' was never erased.")
print(f"   The memory is a whiteboard with no eraser.")

# Check: Germany is still fine (orthogonal key)
retrieved_germany = S_naive.T @ k_germany
print(f"\nQuery Germany: {retrieved_germany.tolist()} (expected {v_berlin.tolist()})")
print(f"Germany OK: {'✅' if torch.allclose(retrieved_germany, v_berlin) else '❌'}")

In [ ]:
#@title 🎧 Listen: What Just Happened
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_09_what_just_happened.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


### What Just Happened?

When we tried to update "France → Lyon", the naive additive rule just **stacked** the new value on top of the old one:

$$S_{\text{new}} = S_{\text{old}} + k_{\text{France}} \cdot v_{\text{Lyon}}^T$$

But $S_{\text{old}}$ already contained $k_{\text{France}} \cdot v_{\text{Paris}}^T$. So when we query:

$$S_{\text{new}}^T k_{\text{France}} = v_{\text{Paris}} + v_{\text{Lyon}}$$

Both values superpose. The memory has no way to distinguish "this is the old value I should forget" from "this is the new value I should remember."

This is the **fundamental weakness** of naive linear attention. It is not just a theoretical concern — it means that as a model processes longer sequences, the state becomes increasingly noisy. Early information interferes with later information, and retrieval quality degrades.

We need a memory that can **correct** itself.

---


In [ ]:
#@title 🎧 Listen: Delta Rule Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_10_delta_rule_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 2. 🎯 The Delta Rule: Error-Correcting Memory

The solution comes from a surprisingly old idea. The **delta rule** was introduced in the 1960s as a learning rule for single-layer neural networks. The insight is elegant:

> **Before writing new information, first check what the memory currently predicts. Then write only the error — the difference between what you want and what is already there.**

Mathematically:

$$S_t = S_{t-1} + k_t \left(v_t - S_{t-1}^T k_t\right)^T$$

Let us break this apart piece by piece:

| Component | Meaning |
|-----------|----------|
| $S_{t-1}^T k_t$ | What the memory **currently predicts** for key $k_t$ |
| $v_t - S_{t-1}^T k_t$ | The **error** — desired value minus current prediction |
| $k_t (\cdot)^T$ | Write the error into memory, indexed by key $k_t$ |

Compare with the naive rule:
- **Naive:** $S_t = S_{t-1} + k_t v_t^T$ (write the raw value)
- **Delta:** $S_t = S_{t-1} + k_t (v_t - S_{t-1}^T k_t)^T$ (write only the correction)

The delta rule turns a dumb accumulator into an **error-correcting associative memory**.

In [ ]:
#@title 🎧 Listen: Delta Rule Worked Example
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_11_delta_rule_worked_example.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


### Worked Example: The 2×2 Matrix

Let us work through the exact numerical example from the article. We start with a state matrix:

$$S_0 = \begin{bmatrix} 0.8 & 0.2 \\ 0.1 & 0.9 \end{bmatrix}$$

A new token arrives with $k_1 = [1, 0]^T$ and $v_1 = [0.5, 0.3]^T$.

**Step 1 — Current prediction:**

$$S_0^T k_1 = \begin{bmatrix} 0.8 & 0.1 \\ 0.2 & 0.9 \end{bmatrix} \begin{bmatrix} 1 \\ 0 \end{bmatrix} = \begin{bmatrix} 0.8 \\ 0.2 \end{bmatrix}$$

**Step 2 — Compute the error:**

$$v_1 - S_0^T k_1 = \begin{bmatrix} 0.5 \\ 0.3 \end{bmatrix} - \begin{bmatrix} 0.8 \\ 0.2 \end{bmatrix} = \begin{bmatrix} -0.3 \\ 0.1 \end{bmatrix}$$

**Step 3 — Update:**

$$S_1 = S_0 + k_1 (v_1 - S_0^T k_1)^T = \begin{bmatrix} 0.8 & 0.2 \\ 0.1 & 0.9 \end{bmatrix} + \begin{bmatrix} 1 \\ 0 \end{bmatrix} \begin{bmatrix} -0.3 & 0.1 \end{bmatrix} = \begin{bmatrix} 0.5 & 0.3 \\ 0.1 & 0.9 \end{bmatrix}$$

**Step 4 — Verify retrieval:**

$$S_1^T k_1 = \begin{bmatrix} 0.5 & 0.1 \\ 0.3 & 0.9 \end{bmatrix} \begin{bmatrix} 1 \\ 0 \end{bmatrix} = \begin{bmatrix} 0.5 \\ 0.3 \end{bmatrix} = v_1 \quad \checkmark$$

The memory retrieves **exactly** the desired value. The old information was surgically corrected, not blindly overwritten.

---


In [ ]:
#@title 🎧 Before You Start: Todo Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_12_todo_1_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 3. 🔧 Your Turn: TODO #1 — Implement the Delta Rule Update

Implement `delta_rule_update(S, k, v)` that computes:

$$S_{\text{new}} = S + k \left(v - S^T k\right)^T$$

Then verify it with the 2×2 numerical example above.

In [ ]:
#@title 🎧 Code Walkthrough: Todo Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_13_todo_1_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# ============ TODO ============
# Implement the delta rule update.
#
# Given:
#   S: state matrix, shape (d_k, d_v)
#   k: key vector, shape (d_k,)
#   v: value vector, shape (d_v,)
#
# Return:
#   S_new: updated state matrix, shape (d_k, d_v)
#
# Formula: S_new = S + k @ (v - S^T @ k)^T
#
# Hint: torch.outer(a, b) computes the outer product a * b^T
# ============ TODO ============

def delta_rule_update(S: torch.Tensor, k: torch.Tensor, v: torch.Tensor) -> torch.Tensor:
    """Apply one step of the delta rule to the state matrix.

    Args:
        S: Current state matrix, shape (d_k, d_v)
        k: Key vector, shape (d_k,)
        v: Value vector, shape (d_v,)

    Returns:
        Updated state matrix S_new, shape (d_k, d_v)
    """
    # Step 1: Current prediction — what does the memory think v should be for this key?
    prediction = ???  # YOUR CODE HERE: S^T @ k

    # Step 2: Error — how far off is the prediction?
    error = ???  # YOUR CODE HERE: v - prediction

    # Step 3: Write the error into the state, indexed by the key
    S_new = ???  # YOUR CODE HERE: S + outer(k, error)

    return S_new

In [ ]:
#@title 🎧 Code Walkthrough: Todo Verification
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_14_todo_1_verification.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# ============ VERIFICATION ============
# Test with the exact numerical example from the article

S_0 = torch.tensor([[0.8, 0.2],
                     [0.1, 0.9]])
k_1 = torch.tensor([1.0, 0.0])
v_1 = torch.tensor([0.5, 0.3])

try:
    S_1 = delta_rule_update(S_0, k_1, v_1)

    expected_S1 = torch.tensor([[0.5, 0.3],
                                [0.1, 0.9]])

    print("--- Worked Example Verification ---")
    print(f"S_0:\n{S_0}")
    print(f"k_1: {k_1.tolist()}, v_1: {v_1.tolist()}")
    print(f"\nS_1 (your result):\n{S_1}")
    print(f"S_1 (expected):\n{expected_S1}")
    assert torch.allclose(S_1, expected_S1, atol=1e-6), "S_1 does not match expected value!"
    print("✅ S_1 matches!")

    # Check retrieval
    retrieved = S_1.T @ k_1
    print(f"\nRetrieval with k_1: {retrieved.tolist()}")
    print(f"Expected v_1:       {v_1.tolist()}")
    assert torch.allclose(retrieved, v_1, atol=1e-6), "Retrieval does not match v_1!"
    print("✅ Retrieval is exact!")

    # Check that the second row (k=[0,1] direction) is unchanged
    k_2_dir = torch.tensor([0.0, 1.0])
    retrieved_2 = S_1.T @ k_2_dir
    retrieved_2_old = S_0.T @ k_2_dir
    print(f"\nRetrieval for k=[0,1] after update:  {retrieved_2.tolist()}")
    print(f"Retrieval for k=[0,1] before update: {retrieved_2_old.tolist()}")
    assert torch.allclose(retrieved_2, retrieved_2_old, atol=1e-6)
    print("✅ Orthogonal direction unaffected!")

    print("\n✅ All checks passed! Your delta rule implementation is correct.")


In [ ]:
#@title 🎧 Listen: After Todo
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_15_after_todo_1.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
except NameError:
    print("❌ Replace the ??? placeholders with your code.")
except Exception as e:
    print(f"❌ Error: {e}")

---


In [ ]:
#@title 🎧 Narration: Stop And Think
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_16_stop_and_think_1.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


### ✋ Stop and Think
Before looking at the solution:
1. Why does the delta rule not disturb the orthogonal direction ($k = [0, 1]$)?
2. What happens if the memory already stores the correct value? What is the error then?
3. What happens if you apply the delta rule update **twice** with the same key and value?

*Take a minute. Then scroll down.*

---

In [ ]:
#@title 🎧 Code Walkthrough: Solution
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_17_solution_1.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


### Solution

In [ ]:
def delta_rule_update(S: torch.Tensor, k: torch.Tensor, v: torch.Tensor) -> torch.Tensor:
    """Apply one step of the delta rule to the state matrix.

    S_new = S + k @ (v - S^T @ k)^T
    """
    # Step 1: Current prediction
    prediction = S.T @ k            # shape: (d_v,)

    # Step 2: Error
    error = v - prediction           # shape: (d_v,)

    # Step 3: Write correction
    S_new = S + torch.outer(k, error)  # shape: (d_k, d_v)

    return S_new


# Verify
S_0 = torch.tensor([[0.8, 0.2],
                     [0.1, 0.9]])
k_1 = torch.tensor([1.0, 0.0])
v_1 = torch.tensor([0.5, 0.3])

S_1 = delta_rule_update(S_0, k_1, v_1)

print(f"S_1 = \n{S_1}")
print(f"Retrieval: S_1^T @ k_1 = {(S_1.T @ k_1).tolist()}")
print(f"Expected:               {v_1.tolist()}")
print(f"\n✅ Delta rule: the memory retrieves exactly v_1.")

---


In [ ]:
#@title 🎧 Transition: Naive Vs Delta Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_18_naive_vs_delta_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 4. 🔬 Naive vs Delta: The Update Showdown

Now let us return to the France/Lyon update scenario and see how the delta rule handles it compared to the naive approach.

In [ ]:
#@title 🎧 Code Walkthrough: Naive Vs Delta Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_19_naive_vs_delta_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
print("=" * 60)
print("  SHOWDOWN: Naive vs Delta Rule on Value Update")
print("=" * 60)

d = 2

# --- Naive Linear Attention ---
S_naive = torch.zeros(d, d)
k_france = torch.tensor([1.0, 0.0])
k_germany = torch.tensor([0.0, 1.0])
v_paris = torch.tensor([0.8, 0.2])
v_berlin = torch.tensor([0.3, 0.9])
v_lyon = torch.tensor([0.6, 0.7])

# Store France -> Paris
S_naive = S_naive + torch.outer(k_france, v_paris)
# Store Germany -> Berlin
S_naive = S_naive + torch.outer(k_germany, v_berlin)
# Update France -> Lyon
S_naive = S_naive + torch.outer(k_france, v_lyon)

retrieved_naive = S_naive.T @ k_france

print(f"\n--- NAIVE Linear Attention ---")
print(f"Query France: {retrieved_naive.tolist()}")
print(f"Expected (Lyon): {v_lyon.tolist()}")
error_naive = torch.norm(retrieved_naive - v_lyon).item()
print(f"Retrieval error (L2): {error_naive:.4f}")
print(f"Result: ❌ Contains superposition of Paris + Lyon")

# --- Delta Rule ---
S_delta = torch.zeros(d, d)

# Store France -> Paris
S_delta = delta_rule_update(S_delta, k_france, v_paris)
# Store Germany -> Berlin
S_delta = delta_rule_update(S_delta, k_germany, v_berlin)
# Update France -> Lyon
S_delta = delta_rule_update(S_delta, k_france, v_lyon)

retrieved_delta = S_delta.T @ k_france

print(f"\n--- DELTA Rule ---")
print(f"Query France: {retrieved_delta.tolist()}")
print(f"Expected (Lyon): {v_lyon.tolist()}")
error_delta = torch.norm(retrieved_delta - v_lyon).item()
print(f"Retrieval error (L2): {error_delta:.6f}")
print(f"Result: ✅ Exact retrieval!")

# Also check Germany is unaffected
retrieved_germany_delta = S_delta.T @ k_germany
print(f"\nQuery Germany (delta): {retrieved_germany_delta.tolist()}")
print(f"Expected (Berlin):     {v_berlin.tolist()}")
print(f"Germany OK: {'✅' if torch.allclose(retrieved_germany_delta, v_berlin, atol=1e-6) else '❌'}")


In [ ]:
#@title 🎧 Listen: After Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_08_after_code_5.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
print(f"\n{'=' * 60}")
print(f"  Naive error: {error_naive:.4f}")
print(f"  Delta error: {error_delta:.6f}")
print(f"  Delta rule eliminated the superposition entirely.")


In [ ]:
#@title 🎧 Listen: After Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_20_after_code_16.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
print(f"{'=' * 60}")

In [ ]:
#@title 🎧 Listen: Whiteboard Analogy
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_21_whiteboard_analogy.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


### The Whiteboard Analogy

Here is a helpful way to think about it:

- **Naive linear attention** is like writing on a whiteboard with no eraser. Every time you learn something new, you write it on the board. But old notes never go away. Eventually the board is covered in overlapping text and you cannot read anything clearly.

- **The delta rule** gives you an eraser — but a smart one. Instead of erasing everything, it reads what is currently written for a particular topic, figures out what needs to change, and makes the minimal correction. Old information that is still correct stays untouched.

Let us now see this difference at scale with a real benchmark.

---


In [ ]:
#@title 🎧 Transition: Full Implementation Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_22_full_implementation_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 5. 🏗️ Building the Full Implementation

Before we benchmark, let us implement both methods as clean, reusable functions that process a full sequence of key-value pairs.

In [ ]:
#@title 🎧 Code Walkthrough: Full Implementation Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_23_full_implementation_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
def naive_linear_attention(keys: torch.Tensor, values: torch.Tensor,
                           queries: torch.Tensor) -> torch.Tensor:
    """Process a sequence with naive additive linear attention.

    Args:
        keys:   (seq_len, d_k) — key vectors to write
        values: (seq_len, d_v) — value vectors to store
        queries: (n_queries, d_k) — queries to retrieve with

    Returns:
        retrieved: (n_queries, d_v) — retrieved values
    """
    seq_len, d_k = keys.shape
    d_v = values.shape[1]

    # Build state by accumulating outer products
    S = torch.zeros(d_k, d_v)
    for t in range(seq_len):
        S = S + torch.outer(keys[t], values[t])

    # Retrieve: output = S^T @ q for each query
    retrieved = queries @ S  # (n_queries, d_k) @ (d_k, d_v) = (n_queries, d_v)
    return retrieved


def deltanet_attention(keys: torch.Tensor, values: torch.Tensor,
                       queries: torch.Tensor) -> torch.Tensor:
    """Process a sequence with delta rule updates.

    Args:
        keys:   (seq_len, d_k) — key vectors to write
        values: (seq_len, d_v) — value vectors to store
        queries: (n_queries, d_k) — queries to retrieve with

    Returns:
        retrieved: (n_queries, d_v) — retrieved values
    """
    seq_len, d_k = keys.shape
    d_v = values.shape[1]

    # Build state with delta rule
    S = torch.zeros(d_k, d_v)
    for t in range(seq_len):
        S = delta_rule_update(S, keys[t], values[t])

    # Retrieve
    retrieved = queries @ S
    return retrieved


# Quick sanity check
keys = torch.tensor([[1.0, 0.0], [0.0, 1.0]])
values = torch.tensor([[0.5, 0.3], [0.7, 0.1]])
queries = torch.tensor([[1.0, 0.0], [0.0, 1.0]])

naive_result = naive_linear_attention(keys, values, queries)
delta_result = deltanet_attention(keys, values, queries)

print("Sanity check (orthogonal keys, no updates):")
print(f"Naive:  {naive_result.tolist()}")
print(f"Delta:  {delta_result.tolist()}")
print(f"Target: {values.tolist()}")
print(f"Both exact: ✅" if torch.allclose(naive_result, values, atol=1e-6) else "Naive differs")

---


In [ ]:
#@title 🎧 Before You Start: Todo Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_24_todo_2_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 6. 🔧 Your Turn: TODO #2 — Build a Key-Value Memory Benchmark

Create a class `AssociativeMemory` that:
1. Generates random key-value pairs (keys are L2-normalized random vectors)
2. Stores them using either naive additive updates or delta rule updates
3. Queries all keys back and measures average retrieval MSE

Test with $N = $ 10, 50, 100, 200 stored pairs and a state dimension of $d = 64$.

In [ ]:
#@title 🎧 Code Walkthrough: Todo Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_25_todo_2_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# ============ TODO ============
# Implement the AssociativeMemory class.
#
# Requirements:
#   - __init__(self, d_k, d_v): Initialize the state S as zeros
#   - write_naive(self, k, v): Update S using naive additive rule
#   - write_delta(self, k, v): Update S using delta rule
#   - read(self, q): Return S^T @ q
#   - reset(self): Reset S to zeros
#
# Then implement benchmark_memory(n_pairs, d_k, d_v, method):
#   - Generate n_pairs random keys (L2-normalized) and values
#   - Write all pairs using the specified method ('naive' or 'delta')
#   - Read back all keys
#   - Return the average MSE between stored values and retrieved values
# ============ TODO ============

class AssociativeMemory:
    """A simple key-value memory using either naive or delta rule updates."""

    def __init__(self, d_k: int, d_v: int):
        self.d_k = d_k
        self.d_v = d_v
        self.S = torch.zeros(d_k, d_v)

    def write_naive(self, k: torch.Tensor, v: torch.Tensor):
        """Additive update: S = S + k @ v^T"""
        ???  # YOUR CODE HERE

    def write_delta(self, k: torch.Tensor, v: torch.Tensor):
        """Delta rule update: S = S + k @ (v - S^T k)^T"""
        ???  # YOUR CODE HERE

    def read(self, q: torch.Tensor) -> torch.Tensor:
        """Query the memory: return S^T @ q"""
        ???  # YOUR CODE HERE

    def reset(self):
        """Reset the state to zeros."""
        self.S = torch.zeros(self.d_k, self.d_v)


def benchmark_memory(n_pairs: int, d_k: int = 64, d_v: int = 64,
                     method: str = 'naive', seed: int = 42) -> float:
    """Benchmark memory retrieval quality.

    Args:
        n_pairs: Number of key-value pairs to store
        d_k: Key dimension
        d_v: Value dimension
        method: 'naive' or 'delta'
        seed: Random seed for reproducibility

    Returns:
        Average MSE across all retrievals
    """
    torch.manual_seed(seed)

    # Generate random keys (L2-normalized) and values
    keys = torch.randn(n_pairs, d_k)
    keys = keys / keys.norm(dim=1, keepdim=True)  # L2 normalize
    values = torch.randn(n_pairs, d_v)

    # Create memory and write all pairs
    mem = AssociativeMemory(d_k, d_v)

    for i in range(n_pairs):
        if method == 'naive':
            mem.write_naive(keys[i], values[i])
        elif method == 'delta':
            mem.write_delta(keys[i], values[i])
        else:
            raise ValueError(f"Unknown method: {method}")

    # Read back all keys and compute MSE
    total_mse = 0.0
    for i in range(n_pairs):
        retrieved = ???  # YOUR CODE HERE: read with keys[i]
        mse = ???  # YOUR CODE HERE: compute MSE between retrieved and values[i]
        total_mse += mse

    return total_mse / n_pairs

In [ ]:
#@title 🎧 Code Walkthrough: Todo Verification
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_26_todo_2_verification.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# ============ VERIFICATION ============

try:
    test_sizes = [10, 50, 100, 200]
    d_k, d_v = 64, 64

    print(f"{'N pairs':<10} {'Naive MSE':>12} {'Delta MSE':>12} {'Winner':>10}")
    print("-" * 48)

    for n in test_sizes:
        mse_naive = benchmark_memory(n, d_k, d_v, method='naive')
        mse_delta = benchmark_memory(n, d_k, d_v, method='delta')
        winner = 'Delta' if mse_delta < mse_naive else 'Naive'
        print(f"{n:<10} {mse_naive:>12.4f} {mse_delta:>12.6f} {winner:>10}")

    # Key assertion: delta should be much better for large N
    mse_naive_200 = benchmark_memory(200, d_k, d_v, method='naive')
    mse_delta_200 = benchmark_memory(200, d_k, d_v, method='delta')
    assert mse_delta_200 < mse_naive_200, "Delta should have lower MSE than naive!"
    print(f"\n✅ Delta rule consistently outperforms naive!")
    print(f"   At N=200: naive MSE is {mse_naive_200/mse_delta_200:.0f}x worse than delta.")


In [ ]:
#@title 🎧 Listen: After Todo
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_27_after_todo_2.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
except NameError:
    print("❌ Replace the ??? placeholders with your code.")
except Exception as e:
    print(f"❌ Error: {e}")

---


In [ ]:
#@title 🎧 Narration: Stop And Think
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_28_stop_and_think_2.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


### ✋ Stop and Think
Before looking at the solution:
1. Why does the naive method work reasonably well for small $N$ but degrade as $N$ grows?
2. When $N > d_k$ (more pairs than the state dimension), what happens to both methods?
3. Can the delta rule achieve **zero** MSE for any number of pairs? Under what condition?

*Take a minute. Then scroll down.*

---

In [ ]:
#@title 🎧 Code Walkthrough: Solution
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_29_solution_2.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


### Solution

In [ ]:
class AssociativeMemory:
    """A simple key-value memory using either naive or delta rule updates."""

    def __init__(self, d_k: int, d_v: int):
        self.d_k = d_k
        self.d_v = d_v
        self.S = torch.zeros(d_k, d_v)

    def write_naive(self, k: torch.Tensor, v: torch.Tensor):
        """Additive update: S = S + k @ v^T"""
        self.S = self.S + torch.outer(k, v)

    def write_delta(self, k: torch.Tensor, v: torch.Tensor):
        """Delta rule update: S = S + k @ (v - S^T k)^T"""
        self.S = delta_rule_update(self.S, k, v)

    def read(self, q: torch.Tensor) -> torch.Tensor:
        """Query the memory: return S^T @ q"""
        return self.S.T @ q

    def reset(self):
        """Reset the state to zeros."""
        self.S = torch.zeros(self.d_k, self.d_v)


def benchmark_memory(n_pairs: int, d_k: int = 64, d_v: int = 64,
                     method: str = 'naive', seed: int = 42) -> float:
    """Benchmark memory retrieval quality."""
    torch.manual_seed(seed)

    # Generate random keys (L2-normalized) and values
    keys = torch.randn(n_pairs, d_k)
    keys = keys / keys.norm(dim=1, keepdim=True)
    values = torch.randn(n_pairs, d_v)

    # Create memory and write all pairs
    mem = AssociativeMemory(d_k, d_v)

    for i in range(n_pairs):
        if method == 'naive':
            mem.write_naive(keys[i], values[i])
        else:
            mem.write_delta(keys[i], values[i])

    # Read back all keys and compute MSE
    total_mse = 0.0
    for i in range(n_pairs):
        retrieved = mem.read(keys[i])
        mse = ((retrieved - values[i]) ** 2).mean().item()
        total_mse += mse

    return total_mse / n_pairs


# Run the benchmark
test_sizes = [10, 50, 100, 200]
d_k, d_v = 64, 64

naive_mses = []
delta_mses = []

print(f"{'N pairs':<10} {'Naive MSE':>12} {'Delta MSE':>12} {'Ratio':>10}")
print("-" * 48)

for n in test_sizes:
    mse_naive = benchmark_memory(n, d_k, d_v, method='naive')
    mse_delta = benchmark_memory(n, d_k, d_v, method='delta')
    naive_mses.append(mse_naive)
    delta_mses.append(mse_delta)
    ratio = mse_naive / max(mse_delta, 1e-10)
    print(f"{n:<10} {mse_naive:>12.4f} {mse_delta:>12.6f} {ratio:>9.0f}x")

print(f"\n✅ Delta rule is dramatically better at all scales.")
print(f"   As N grows, naive MSE explodes while delta stays controlled.")

---


In [ ]:
#@title 🎧 Before You Start: Todo Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_30_todo_3_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 7. 📊 Visualization: Memory Quality

### TODO #3 — Visualize Memory Degradation

Create two visualizations:

1. **Retrieval error (MSE) vs number of stored items** for naive vs delta (line plot)
2. **Singular values of the state matrix** $S$ for both methods after storing 100 pairs (bar chart)

The singular value plot reveals *why* the delta rule works: it keeps the state matrix "cleaner" with a more concentrated spectrum.

In [ ]:
#@title 🎧 Code Walkthrough: Todo Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_31_todo_3_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# ============ TODO ============
# Create two visualizations:
#
# Plot 1: Retrieval MSE vs N for both methods
#   - X axis: number of stored pairs (use range 5 to 250 in steps of 5)
#   - Y axis: average retrieval MSE
#   - Two lines: one for naive (red), one for delta (blue)
#   - Use d_k = d_v = 64
#   - Add a vertical dashed line at N = d_k = 64 (the "capacity" limit)
#
# Plot 2: Singular values of S after storing 100 pairs
#   - Build S_naive and S_delta by storing 100 random pairs
#   - Compute SVD of each: torch.linalg.svdvals(S)
#   - Bar chart: singular values sorted descending
#   - Two sets of bars: naive (red) vs delta (blue)
#
# Use fig, axes = plt.subplots(1, 2, figsize=(16, 5))
# ============ TODO ============

# --- Data collection for Plot 1 ---
d_k, d_v = 64, 64
n_range = list(range(5, 255, 5))

naive_curve = []
delta_curve = []

for n in n_range:
    naive_curve.append(benchmark_memory(n, d_k, d_v, method='naive'))
    delta_curve.append(benchmark_memory(n, d_k, d_v, method='delta'))

# --- Data collection for Plot 2 ---
torch.manual_seed(42)
n_store = 100
keys_100 = torch.randn(n_store, d_k)
keys_100 = keys_100 / keys_100.norm(dim=1, keepdim=True)
values_100 = torch.randn(n_store, d_v)

# Build both state matrices
S_naive_100 = torch.zeros(d_k, d_v)
S_delta_100 = torch.zeros(d_k, d_v)

for i in range(n_store):
    S_naive_100 = S_naive_100 + torch.outer(keys_100[i], values_100[i])
    S_delta_100 = delta_rule_update(S_delta_100, keys_100[i], values_100[i])

sv_naive = torch.linalg.svdvals(S_naive_100).numpy()
sv_delta = torch.linalg.svdvals(S_delta_100).numpy()

# --- Plotting ---
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Plot 1: MSE vs N
???  # YOUR CODE HERE
# - axes[0].plot(...) for naive (color='#e53935', label='Naive Linear Attention')
# - axes[0].plot(...) for delta (color='#1565c0', label='DeltaNet (Delta Rule)')
# - axes[0].axvline(x=d_k, ...) for capacity line
# - Add labels, title, legend

# Plot 2: Singular values
???  # YOUR CODE HERE
# - Bar chart comparing sv_naive and sv_delta
# - x = np.arange(len(sv_naive)), width = 0.35
# - axes[1].bar(x - width/2, sv_naive, width, ...) for naive
# - axes[1].bar(x + width/2, sv_delta, width, ...) for delta
# - Only show first 20 singular values for readability

plt.tight_layout()
plt.show()

---


In [ ]:
#@title 🎧 Narration: Stop And Think
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_32_stop_and_think_3.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


### ✋ Stop and Think
Before looking at the solution:
1. At what value of $N$ does naive linear attention start to clearly degrade? Why is $N = d_k$ a critical threshold?
2. What does it mean for the singular values to be more "spread out" vs "concentrated"?
3. Why might a flatter singular value distribution indicate better memory utilization?

*Take a minute. Then scroll down.*

---

In [ ]:
#@title 🎧 Code Walkthrough: Solution
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_33_solution_3.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


### Solution

In [ ]:
# --- Data collection for Plot 1 ---
d_k, d_v = 64, 64
n_range = list(range(5, 255, 5))

naive_curve = []
delta_curve = []

for n in n_range:
    naive_curve.append(benchmark_memory(n, d_k, d_v, method='naive'))
    delta_curve.append(benchmark_memory(n, d_k, d_v, method='delta'))

# --- Data collection for Plot 2 ---
torch.manual_seed(42)
n_store = 100
keys_100 = torch.randn(n_store, d_k)
keys_100 = keys_100 / keys_100.norm(dim=1, keepdim=True)
values_100 = torch.randn(n_store, d_v)

S_naive_100 = torch.zeros(d_k, d_v)
S_delta_100 = torch.zeros(d_k, d_v)

for i in range(n_store):
    S_naive_100 = S_naive_100 + torch.outer(keys_100[i], values_100[i])
    S_delta_100 = delta_rule_update(S_delta_100, keys_100[i], values_100[i])

sv_naive = torch.linalg.svdvals(S_naive_100).numpy()
sv_delta = torch.linalg.svdvals(S_delta_100).numpy()

# --- Plotting ---
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Plot 1: MSE vs N
axes[0].plot(n_range, naive_curve, color='#e53935', linewidth=2.5,
             label='Naive Linear Attention', marker='o', markersize=3)
axes[0].plot(n_range, delta_curve, color='#1565c0', linewidth=2.5,
             label='DeltaNet (Delta Rule)', marker='s', markersize=3)
axes[0].axvline(x=d_k, color='gray', linestyle='--', linewidth=1.5,
                label=f'State dimension d={d_k}')
axes[0].set_xlabel('Number of Stored Pairs (N)', fontsize=12)
axes[0].set_ylabel('Average Retrieval MSE', fontsize=12)
axes[0].set_title('Retrieval Error vs Memory Load', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=10)
axes[0].set_ylim(bottom=-0.05)

# Plot 2: Singular values (first 20)
n_show = 20
x = np.arange(n_show)
width = 0.35

axes[1].bar(x - width/2, sv_naive[:n_show], width, color='#e53935',
            alpha=0.8, label='Naive', edgecolor='white')
axes[1].bar(x + width/2, sv_delta[:n_show], width, color='#1565c0',
            alpha=0.8, label='DeltaNet', edgecolor='white')
axes[1].set_xlabel('Singular Value Index', fontsize=12)
axes[1].set_ylabel('Singular Value', fontsize=12)
axes[1].set_title(f'State Matrix Singular Values (N={n_store})', fontsize=14,
                  fontweight='bold')
axes[1].legend(fontsize=11)
axes[1].set_xticks(x[::2])

plt.tight_layout()
plt.show()

print("Left: Naive MSE grows roughly linearly with N; DeltaNet stays much lower.")
print(f"Right: DeltaNet's singular values are more concentrated — the state is 'cleaner'.")
print(f"       Naive has a few very large singular values and many small ones (noise spread).")

In [ ]:
#@title 🎧 Listen: Interpreting Singular Values
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_34_interpreting_singular_values.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


### Interpreting the Singular Values

The singular value decomposition (SVD) of the state matrix $S$ tells us how the stored information is distributed across the matrix's dimensions.

- **Naive linear attention**: The singular values are spread out unevenly. A few dominant directions accumulate most of the energy, while others get noise. This is because the additive updates keep piling on information without any correction. Random key-value pairs create a "pile-up" where some directions in the state space get saturated.

- **Delta rule**: The singular values are more controlled. The error-correction mechanism actively adjusts the state so that stored information is distributed more efficiently. When a key already has a good representation, the delta rule makes only a small correction, avoiding unnecessary amplification.

In other words, the delta rule uses the state matrix's **capacity** more efficiently. It is not wasting dimensions on redundant or contradictory information.

---


In [ ]:
#@title 🎧 Transition: Scaling Behavior Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_35_scaling_behavior_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 8. 📈 Scaling Behavior: What Happens at Large N?

Let us push the benchmark further and see how both methods scale as we go well beyond the state dimension.

In [ ]:
#@title 🎧 What to Look For: Scaling Behavior Code Viz
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_36_scaling_behavior_code_viz.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Extended scaling experiment
d_k, d_v = 64, 64
n_extended = [10, 25, 50, 75, 100, 150, 200, 300, 400, 500]

naive_extended = []
delta_extended = []

print(f"{'N':>6} {'Naive MSE':>12} {'Delta MSE':>12} {'Ratio':>10}")
print("-" * 44)

for n in n_extended:
    mn = benchmark_memory(n, d_k, d_v, method='naive')
    md = benchmark_memory(n, d_k, d_v, method='delta')
    naive_extended.append(mn)
    delta_extended.append(md)
    ratio = mn / max(md, 1e-10)
    print(f"{n:>6} {mn:>12.4f} {md:>12.6f} {ratio:>9.0f}x")

# Plot
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Linear scale
axes[0].plot(n_extended, naive_extended, 'o-', color='#e53935', linewidth=2,
             markersize=6, label='Naive')
axes[0].plot(n_extended, delta_extended, 's-', color='#1565c0', linewidth=2,
             markersize=6, label='DeltaNet')
axes[0].axvline(x=d_k, color='gray', linestyle='--', alpha=0.7, label=f'd={d_k}')
axes[0].set_xlabel('Number of Stored Pairs (N)', fontsize=12)
axes[0].set_ylabel('Avg Retrieval MSE', fontsize=12)
axes[0].set_title('Linear Scale', fontsize=13, fontweight='bold')
axes[0].legend(fontsize=10)

# Log scale
axes[1].semilogy(n_extended, naive_extended, 'o-', color='#e53935', linewidth=2,
                 markersize=6, label='Naive')
axes[1].semilogy(n_extended, delta_extended, 's-', color='#1565c0', linewidth=2,
                 markersize=6, label='DeltaNet')
axes[1].axvline(x=d_k, color='gray', linestyle='--', alpha=0.7, label=f'd={d_k}')
axes[1].set_xlabel('Number of Stored Pairs (N)', fontsize=12)
axes[1].set_ylabel('Avg Retrieval MSE (log)', fontsize=12)
axes[1].set_title('Log Scale (reveals delta behavior)', fontsize=13, fontweight='bold')
axes[1].legend(fontsize=10)

plt.suptitle('Scaling Behavior: Naive vs DeltaNet', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f"\nKey observations:")
print(f"  1. Naive MSE grows roughly as O(N) — every new pair adds noise.")
print(f"  2. DeltaNet MSE stays much lower but does eventually grow when N >> d.")
print(f"  3. The gap between them widens as N increases.")
print(f"  4. Neither method achieves zero error for N > d (the state has finite capacity).")

---


In [ ]:
#@title 🎧 Transition: Orthogonal Keys Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_37_orthogonal_keys_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 9. 🧪 Special Case: Orthogonal Keys

There is one scenario where even naive linear attention works perfectly: when all keys are **orthogonal** to each other. Let us see why, and what this tells us about the delta rule's advantage.

In [ ]:
#@title 🎧 Code Walkthrough: Orthogonal Keys Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_38_orthogonal_keys_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Special case: orthogonal keys
d = 32

# Generate exactly d orthogonal keys using QR decomposition
torch.manual_seed(42)
random_matrix = torch.randn(d, d)
Q, _ = torch.linalg.qr(random_matrix)  # Q is orthogonal: Q^T Q = I
ortho_keys = Q  # Each row is a unit-length, mutually orthogonal key

# Random values
ortho_values = torch.randn(d, d)

# Test naive
S_naive = torch.zeros(d, d)
for i in range(d):
    S_naive = S_naive + torch.outer(ortho_keys[i], ortho_values[i])

# Test delta
S_delta = torch.zeros(d, d)
for i in range(d):
    S_delta = delta_rule_update(S_delta, ortho_keys[i], ortho_values[i])

# Measure MSE
naive_mse_ortho = 0.0
delta_mse_ortho = 0.0
for i in range(d):
    ret_n = S_naive.T @ ortho_keys[i]
    ret_d = S_delta.T @ ortho_keys[i]
    naive_mse_ortho += ((ret_n - ortho_values[i]) ** 2).mean().item()
    delta_mse_ortho += ((ret_d - ortho_values[i]) ** 2).mean().item()

naive_mse_ortho /= d
delta_mse_ortho /= d

print(f"Orthogonal keys (N = d = {d}):")
print(f"  Naive MSE: {naive_mse_ortho:.8f}")
print(f"  Delta MSE: {delta_mse_ortho:.8f}")
print(f"\n✅ Both achieve near-zero error with orthogonal keys!")
print(f"\nWhy? With orthogonal keys, k_i^T k_j = 0 for i ≠ j.")
print(f"Each key writes to an independent 'slot' in the matrix.")
print(f"No interference between writes = no error to correct.")
print(f"\nBut real keys in neural networks are NOT orthogonal.")
print(f"They are learned from data and often have significant overlap.")
print(f"This is exactly where the delta rule shines.")

In [ ]:
#@title 🎧 What to Look For: Key Overlap Demo
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_39_key_overlap_demo.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Demonstrate: how key overlap affects retrieval quality
d_k, d_v = 64, 64
n_pairs = 100

overlap_levels = np.linspace(0.0, 0.9, 10)  # 0 = random, 0.9 = very similar keys

naive_overlap_mses = []
delta_overlap_mses = []

for overlap in overlap_levels:
    torch.manual_seed(42)

    # Create keys with controlled overlap:
    # Each key = (1-overlap) * random_component + overlap * shared_direction
    shared = torch.randn(d_k)
    shared = shared / shared.norm()

    keys = torch.randn(n_pairs, d_k)
    keys = (1 - overlap) * keys + overlap * shared.unsqueeze(0)
    keys = keys / keys.norm(dim=1, keepdim=True)  # Normalize

    values = torch.randn(n_pairs, d_v)

    # Naive
    S_n = torch.zeros(d_k, d_v)
    for i in range(n_pairs):
        S_n = S_n + torch.outer(keys[i], values[i])

    # Delta
    S_d = torch.zeros(d_k, d_v)
    for i in range(n_pairs):
        S_d = delta_rule_update(S_d, keys[i], values[i])

    # Measure
    n_mse, d_mse = 0.0, 0.0
    for i in range(n_pairs):
        n_mse += ((S_n.T @ keys[i] - values[i]) ** 2).mean().item()
        d_mse += ((S_d.T @ keys[i] - values[i]) ** 2).mean().item()

    naive_overlap_mses.append(n_mse / n_pairs)
    delta_overlap_mses.append(d_mse / n_pairs)

# Plot
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(overlap_levels, naive_overlap_mses, 'o-', color='#e53935', linewidth=2.5,
        markersize=7, label='Naive')
ax.plot(overlap_levels, delta_overlap_mses, 's-', color='#1565c0', linewidth=2.5,
        markersize=7, label='DeltaNet')
ax.set_xlabel('Key Overlap (0 = random, 0.9 = very similar)', fontsize=12)
ax.set_ylabel('Average Retrieval MSE', fontsize=12)
ax.set_title(f'Effect of Key Overlap on Retrieval Quality (N={n_pairs}, d={d_k})',
             fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

print("As keys become more similar (higher overlap), naive performance degrades dramatically.")
print("The delta rule is much more robust to key overlap — exactly the scenario in real models.")

---


In [ ]:
#@title 🎧 Transition: Visualizing State Matrix Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_40_visualizing_state_matrix_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 10. 🎨 Visualizing the State Matrix

Let us look directly at the state matrices to see the structural difference between naive and delta rule updates.

In [ ]:
#@title 🎧 What to Look For: Visualizing State Matrix Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_41_visualizing_state_matrix_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Build state matrices for visualization (using smaller d for clarity)
torch.manual_seed(42)
d_viz = 16
n_viz = 50

keys_viz = torch.randn(n_viz, d_viz)
keys_viz = keys_viz / keys_viz.norm(dim=1, keepdim=True)
values_viz = torch.randn(n_viz, d_viz)

S_naive_viz = torch.zeros(d_viz, d_viz)
S_delta_viz = torch.zeros(d_viz, d_viz)

for i in range(n_viz):
    S_naive_viz = S_naive_viz + torch.outer(keys_viz[i], values_viz[i])
    S_delta_viz = delta_rule_update(S_delta_viz, keys_viz[i], values_viz[i])

# Heatmaps
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Naive state
vmax = max(abs(S_naive_viz).max().item(), abs(S_delta_viz).max().item())
im0 = axes[0].imshow(S_naive_viz.numpy(), cmap='RdBu_r', vmin=-vmax, vmax=vmax)
axes[0].set_title(f'Naive State Matrix\n(N={n_viz}, d={d_viz})', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Value dimension')
axes[0].set_ylabel('Key dimension')
plt.colorbar(im0, ax=axes[0], shrink=0.8)

# Delta state
im1 = axes[1].imshow(S_delta_viz.numpy(), cmap='RdBu_r', vmin=-vmax, vmax=vmax)
axes[1].set_title(f'DeltaNet State Matrix\n(N={n_viz}, d={d_viz})', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Value dimension')
axes[1].set_ylabel('Key dimension')
plt.colorbar(im1, ax=axes[1], shrink=0.8)

# Difference
diff = S_naive_viz - S_delta_viz
im2 = axes[2].imshow(diff.numpy(), cmap='RdBu_r',
                      vmin=-abs(diff).max().item(), vmax=abs(diff).max().item())
axes[2].set_title('Difference (Naive - Delta)', fontsize=13, fontweight='bold')
axes[2].set_xlabel('Value dimension')
axes[2].set_ylabel('Key dimension')
plt.colorbar(im2, ax=axes[2], shrink=0.8)

plt.suptitle('State Matrix Structure: Naive vs DeltaNet', fontsize=15,
             fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f"Naive state Frobenius norm:  {torch.norm(S_naive_viz).item():.2f}")
print(f"DeltaNet state Frobenius norm: {torch.norm(S_delta_viz).item():.2f}")
print(f"Difference norm:               {torch.norm(diff).item():.2f}")
print(f"\nThe naive state has larger magnitude — accumulated noise inflates the entries.")
print(f"The delta state is more compact — it only stores what is needed.")

---


In [ ]:
#@title 🎧 Transition: Progressive Memory Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_42_progressive_memory_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 11. 🎬 Progressive Memory: Watching the State Evolve

Let us watch how the retrieval error evolves **as we write** more key-value pairs, one at a time.

In [ ]:
#@title 🎧 What to Look For: Progressive Memory Code Viz
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_43_progressive_memory_code_viz.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Track retrieval error after each write
torch.manual_seed(42)
d_k, d_v = 64, 64
n_total = 200

keys = torch.randn(n_total, d_k)
keys = keys / keys.norm(dim=1, keepdim=True)
values = torch.randn(n_total, d_v)

S_naive = torch.zeros(d_k, d_v)
S_delta = torch.zeros(d_k, d_v)

naive_errors = []  # error on the MOST RECENT write
delta_errors = []
naive_avg_errors = []  # average error over ALL stored pairs
delta_avg_errors = []

for t in range(n_total):
    # Write
    S_naive = S_naive + torch.outer(keys[t], values[t])
    S_delta = delta_rule_update(S_delta, keys[t], values[t])

    # Error on the just-written pair
    ret_naive = S_naive.T @ keys[t]
    ret_delta = S_delta.T @ keys[t]
    naive_errors.append(((ret_naive - values[t]) ** 2).mean().item())
    delta_errors.append(((ret_delta - values[t]) ** 2).mean().item())

    # Average error over all pairs so far
    if (t + 1) % 10 == 0 or t == 0:  # Check every 10 steps for speed
        n_avg, d_avg = 0.0, 0.0
        for i in range(t + 1):
            n_avg += ((S_naive.T @ keys[i] - values[i]) ** 2).mean().item()
            d_avg += ((S_delta.T @ keys[i] - values[i]) ** 2).mean().item()
        naive_avg_errors.append((t + 1, n_avg / (t + 1)))
        delta_avg_errors.append((t + 1, d_avg / (t + 1)))

# Plot
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Left: error on each newly written pair
axes[0].plot(range(1, n_total + 1), naive_errors, color='#e53935', alpha=0.6,
             linewidth=1, label='Naive')
axes[0].plot(range(1, n_total + 1), delta_errors, color='#1565c0', alpha=0.6,
             linewidth=1, label='DeltaNet')
axes[0].axvline(x=d_k, color='gray', linestyle='--', alpha=0.7, label=f'd={d_k}')
axes[0].set_xlabel('Write Step', fontsize=12)
axes[0].set_ylabel('MSE on Just-Written Pair', fontsize=12)
axes[0].set_title('Immediate Retrieval Error After Each Write', fontsize=13,
                  fontweight='bold')
axes[0].legend(fontsize=10)

# Right: average error over all pairs
n_steps, n_vals = zip(*naive_avg_errors)
d_steps, d_vals = zip(*delta_avg_errors)
axes[1].plot(n_steps, n_vals, 'o-', color='#e53935', linewidth=2, markersize=4,
             label='Naive')
axes[1].plot(d_steps, d_vals, 's-', color='#1565c0', linewidth=2, markersize=4,
             label='DeltaNet')
axes[1].axvline(x=d_k, color='gray', linestyle='--', alpha=0.7, label=f'd={d_k}')
axes[1].set_xlabel('Total Pairs Stored', fontsize=12)
axes[1].set_ylabel('Avg MSE Over All Stored Pairs', fontsize=12)
axes[1].set_title('Global Memory Quality Over Time', fontsize=13, fontweight='bold')
axes[1].legend(fontsize=10)

plt.suptitle('How Memory Degrades Over Time', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("Left: For naive, even the JUST-WRITTEN pair has error (because old noise interferes).")
print("      DeltaNet achieves near-zero error on each immediate write.")
print("Right: Global average tells the full story — naive degrades steadily, delta stays controlled.")

---


In [ ]:
#@title 🎧 Listen: Delta Rule Alone Not Enough Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_44_delta_rule_alone_not_enough_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 12. ⚠️ Why the Delta Rule Alone Is Not Enough

We have seen that the delta rule is dramatically better than naive additive updates. But is it sufficient for a real language model? Let us explore its limitations.

### Limitation 1: No Full Forgetting

The delta rule can *correct* values associated with a key, but it cannot *completely erase* information. If the model needs to entirely forget a context — for example, when processing a new paragraph that has nothing to do with the previous one — the delta rule cannot help. It can only adjust what is already there; it cannot set the memory to zero.

### Limitation 2: Interference with Non-Orthogonal Keys

When keys have significant overlap, updating one key's value slightly affects what gets retrieved for nearby keys. The delta rule minimizes this but cannot eliminate it entirely.

In [ ]:
#@title 🎧 Code Walkthrough: Delta Rule Cannot Fully Forget Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_45_delta_rule_cannot_fully_forget_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Demonstration: the delta rule cannot fully forget
print("=" * 60)
print("  LIMITATION: Delta Rule Cannot Fully Forget")
print("=" * 60)

d = 4
S = torch.zeros(d, d)

# Store a "context" worth of information
torch.manual_seed(42)
context_keys = torch.randn(3, d)
context_keys = context_keys / context_keys.norm(dim=1, keepdim=True)
context_values = torch.randn(3, d)

for i in range(3):
    S = delta_rule_update(S, context_keys[i], context_values[i])

print(f"\nState after storing 3 key-value pairs:")
print(f"Frobenius norm: {torch.norm(S).item():.4f}")
print(f"S =\n{S}")

# Now we want to COMPLETELY FORGET this context.
# The delta rule can set specific key-value associations to zero...
# by writing v=0 for each key:
for i in range(3):
    S = delta_rule_update(S, context_keys[i], torch.zeros(d))

print(f"\nState after writing v=0 for all 3 keys:")
print(f"Frobenius norm: {torch.norm(S).item():.4f}")
print(f"S =\n{S}")

# Check: are the associations cleared?
for i in range(3):
    ret = S.T @ context_keys[i]
    print(f"  Key {i} retrieval: norm = {torch.norm(ret).item():.6f}")

print(f"\n✅ Specific associations are cleared (retrieval is near-zero).")
print(f"⚠️ BUT the state matrix is NOT zero! Residual structure remains.")
print(f"   Frobenius norm: {torch.norm(S).item():.4f}")
print(f"\nThis means queries with NEW keys that overlap with the old keys")
print(f"may still pick up traces of the old context.")

# Demonstrate residual interference
new_key = torch.randn(d)
new_key = new_key / new_key.norm()
residual = S.T @ new_key
print(f"\nNew random key retrieval norm: {torch.norm(residual).item():.4f}")
print(f"(Should be 0 if memory were truly empty, but it is {torch.norm(residual).item():.4f})")

In [ ]:
#@title 🎧 Code Walkthrough: Gating Preview Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_46_gating_preview_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# What we WANT: a decay gate that can clear the memory
print("=" * 60)
print("  PREVIEW: What Gating Gives Us (Notebook 3)")
print("=" * 60)

# Gated DeltaNet update:
# S_t = alpha * S_{t-1} + beta * k_t (v_t - S_{t-1}^T k_t)^T
#
# When alpha = 0: TOTAL MEMORY RESET
# When alpha = 1: Pure delta rule (what we have now)
# When 0 < alpha < 1: Partial decay + correction

def gated_delta_update(S, k, v, alpha, beta):
    """Gated DeltaNet update (preview — we build this fully in Notebook 3)."""
    prediction = S.T @ k
    error = v - prediction
    S_new = alpha * S + beta * torch.outer(k, error)
    return S_new

# Demonstrate: alpha=0 gives complete memory reset
d = 4
torch.manual_seed(42)

# Build up some state
S = torch.zeros(d, d)
for _ in range(10):
    k = torch.randn(d)
    k = k / k.norm()
    v = torch.randn(d)
    S = delta_rule_update(S, k, v)

print(f"\nState after 10 writes (delta rule):")
print(f"  Frobenius norm: {torch.norm(S).item():.4f}")

# Apply gated update with alpha near 0 (aggressive forget)
k_new = torch.randn(d)
k_new = k_new / k_new.norm()
v_new = torch.randn(d)

for alpha_val in [1.0, 0.5, 0.1, 0.0]:
    S_gated = gated_delta_update(S, k_new, v_new, alpha=alpha_val, beta=1.0)
    print(f"\n  alpha={alpha_val:.1f}: Frobenius norm = {torch.norm(S_gated).item():.4f}")
    if alpha_val == 0.0:
        print(f"           Memory is now essentially just the new key-value pair.")

print(f"\n✨ Gating gives the model a 'master switch' for forgetting.")
print(f"   alpha ≈ 0: Forget everything, start fresh.")
print(f"   alpha ≈ 1: Keep everything, just correct (= delta rule).")
print(f"   The model LEARNS when to use each extreme.")
print(f"\n   This is exactly what Gated DeltaNet does in Qwen3.5.")
print(f"   We will build the full implementation in Notebook 3.")

---


In [ ]:
#@title 🎧 Transition: Summary Dashboard Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_47_summary_dashboard_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 13. 📊 Summary Dashboard

Let us bring everything together in one comprehensive visualization.

In [ ]:
#@title 🎧 What to Look For: Summary Dashboard Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_48_summary_dashboard_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Final comprehensive comparison
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('DeltaNet vs Naive Linear Attention: Complete Comparison',
             fontsize=16, fontweight='bold', y=1.01)

# --- Panel 1: MSE vs N (from extended benchmark) ---
axes[0, 0].plot(n_extended, naive_extended, 'o-', color='#e53935', linewidth=2,
                markersize=5, label='Naive')
axes[0, 0].plot(n_extended, delta_extended, 's-', color='#1565c0', linewidth=2,
                markersize=5, label='DeltaNet')
axes[0, 0].axvline(x=64, color='gray', linestyle='--', alpha=0.7, label='d=64')
axes[0, 0].set_xlabel('N (stored pairs)', fontsize=11)
axes[0, 0].set_ylabel('Avg MSE', fontsize=11)
axes[0, 0].set_title('Retrieval Error vs Memory Load', fontsize=13, fontweight='bold')
axes[0, 0].legend(fontsize=9)

# --- Panel 2: Key overlap effect ---
axes[0, 1].plot(overlap_levels, naive_overlap_mses, 'o-', color='#e53935',
                linewidth=2, markersize=5, label='Naive')
axes[0, 1].plot(overlap_levels, delta_overlap_mses, 's-', color='#1565c0',
                linewidth=2, markersize=5, label='DeltaNet')
axes[0, 1].set_xlabel('Key Overlap', fontsize=11)
axes[0, 1].set_ylabel('Avg MSE', fontsize=11)
axes[0, 1].set_title('Effect of Key Similarity', fontsize=13, fontweight='bold')
axes[0, 1].legend(fontsize=9)

# --- Panel 3: Singular values ---
n_show = 20
x = np.arange(n_show)
width = 0.35
axes[1, 0].bar(x - width/2, sv_naive[:n_show], width, color='#e53935',
               alpha=0.8, label='Naive', edgecolor='white')
axes[1, 0].bar(x + width/2, sv_delta[:n_show], width, color='#1565c0',
               alpha=0.8, label='DeltaNet', edgecolor='white')
axes[1, 0].set_xlabel('Singular Value Index', fontsize=11)
axes[1, 0].set_ylabel('Singular Value', fontsize=11)
axes[1, 0].set_title('State Matrix Spectrum (N=100)', fontsize=13, fontweight='bold')
axes[1, 0].legend(fontsize=9)
axes[1, 0].set_xticks(x[::2])

# --- Panel 4: Progressive error ---
axes[1, 1].plot(range(1, n_total + 1), naive_errors, color='#e53935', alpha=0.5,
                linewidth=1, label='Naive')
axes[1, 1].plot(range(1, n_total + 1), delta_errors, color='#1565c0', alpha=0.5,
                linewidth=1, label='DeltaNet')
axes[1, 1].axvline(x=64, color='gray', linestyle='--', alpha=0.7, label='d=64')
axes[1, 1].set_xlabel('Write Step', fontsize=11)
axes[1, 1].set_ylabel('Immediate Write MSE', fontsize=11)
axes[1, 1].set_title('Retrieval Error Per Write Over Time', fontsize=13,
                      fontweight='bold')
axes[1, 1].legend(fontsize=9)

plt.tight_layout()
plt.show()

print("\nTop-left:     DeltaNet scales much better as memory load increases.")
print("Top-right:    DeltaNet is robust to key overlap; naive collapses.")
print("Bottom-left:  DeltaNet keeps a cleaner state (more concentrated singular values).")
print("Bottom-right: DeltaNet achieves near-zero error on each immediate write.")

---


In [ ]:
#@title 🎧 Listen: Reflection Takeaways
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_49_reflection_takeaways.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 14. 🤔 Reflection and Key Takeaways

### What We Learned

1. **Naive linear attention ($S_t = S_{t-1} + k_t v_t^T$) is a whiteboard with no eraser.** Every write permanently adds to the state. Old values superpose with new ones, and retrieval quality degrades as more information is stored.

2. **The delta rule ($S_t = S_{t-1} + k_t(v_t - S_{t-1}^T k_t)^T$) adds an eraser.** Before writing, it checks what the memory currently predicts and writes only the *error*. This turns a dumb accumulator into an error-correcting associative memory.

3. **The improvement is dramatic and consistent.** In our benchmarks, DeltaNet achieved orders-of-magnitude lower retrieval error than naive linear attention, especially as the number of stored pairs grows.

4. **The delta rule works by using the state matrix's capacity more efficiently.** The singular value analysis showed that DeltaNet keeps the state matrix "cleaner" with a more concentrated spectrum.

5. **But the delta rule has limitations.** It cannot completely erase information — only correct specific key-value associations. For wholesale memory clearing, we need **gating** (a learnable decay parameter $\alpha_t$ that controls how much of the old state to retain).

### The Connection to Qwen3.5

The delta rule is **one of the two core innovations** in the Gated DeltaNet layer that powers 75% of Qwen3.5's layers. The other is exponential gating, which adds the ability to selectively forget. Together:

$$S_t = \alpha_t \cdot S_{t-1} + \beta_t \cdot k_t (v_t - S_{t-1}^T k_t)^T$$

- **Gating** ($\alpha_t$): Controls *how much* to remember vs forget (wholesale erasure)
- **Delta rule**: Controls *what* to write (surgical correction)

### Reflection Questions

1. **Why is the delta rule dated from the 1960s, yet only recently applied to attention mechanisms?** Think about what had to change in the field for this old idea to become relevant again.

2. **In a real Transformer, are keys likely to be orthogonal?** What does this tell you about the practical importance of the delta rule?

3. **The delta rule requires $O(d^2)$ computation per step (matrix-vector multiply for the prediction). Is this faster or slower than standard softmax attention?** When does it win?

### Optional Challenges

1. **Multi-step convergence:** Apply the delta rule to the same key-value pair $N$ times in a row. How many iterations does it take to converge to exact retrieval? How does this change with key overlap?

2. **Non-normalized keys:** Remove the L2 normalization from the keys. How does this affect both methods? Why does the delta rule become unstable without normalized keys?

3. **Capacity analysis:** For a $d \times d$ state matrix, what is the theoretical maximum number of orthogonal key-value pairs that can be stored exactly? Can the delta rule match this limit?

---


In [ ]:
#@title 🎧 Transition: What Comes Next
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_50_what_comes_next.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 15. 🚀 What Comes Next

In **Notebook 3: Gated DeltaNet — The Full Layer**, we will:

1. Add exponential gating ($\alpha_t$, $\beta_t$) for selective memory decay
2. Implement the full Gated DeltaNet layer with Conv1D, SiLU, L2 normalization
3. Build the multi-head structure where each head maintains an independent memory
4. Benchmark the complete layer against standard attention on a language modeling task

The delta rule gave us the eraser. Gating gives us the ability to choose *when* and *how aggressively* to use it. Together, they form the backbone of Qwen3.5's efficiency.

In [ ]:
#@title 🎧 Wrap-Up: Closing
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_51_closing.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Final summary
print("""
╔════════════════════════════════════════════════════════════╗
║           DELTANET CHEAT SHEET                        ║
╠════════════════════════════════════════════════════════════╣
║                                                            ║
║  Naive:  S_t = S_{t-1} + k_t v_t^T                        ║
║          (additive only — no correction)                   ║
║                                                            ║
║  Delta:  S_t = S_{t-1} + k_t (v_t - S_{t-1}^T k_t)^T     ║
║          (error-correcting — write only the residual)     ║
║                                                            ║
║  Gated:  S_t = a_t * S_{t-1}                               ║
║               + b_t * k_t (v_t - S_{t-1}^T k_t)^T         ║
║          (decay + correction — Notebook 3)                ║
║                                                            ║
║  Key Insight:                                               ║
║    Naive = whiteboard, no eraser                            ║
║    Delta = whiteboard + smart eraser                        ║
║    Gated = whiteboard + eraser + memory wipe button         ║
║                                                            ║
╚════════════════════════════════════════════════════════════╝
""")
print("✅ Notebook complete! You now understand the delta rule and why it matters.")
print("📊 Next up: Notebook 3 — Gated DeltaNet: The Full Layer")